# 05 — Train với dữ liệu người bản ngữ, và lần đầu đo văn bản DÀI

Notebook 03/04 train **chỉ trên cụm từ điển** (trung vị 4 âm tiết). Giờ có thêm
**996 câu + 115 đoạn** do người bản ngữ dịch.

| Tập | Nguồn | Số lượng | Vai trò |
|---|---|---|---|
| Train | cụm từ điển | 10.576 | như cũ |
| Train | **câu người bản ngữ dịch** | **996** | mới |
| Train | **đoạn văn** | **115** | mới |
| Dev-câu | 226 cặp cũ | 226 | so trực tiếp với 34,85 |
| **Dev-dài** | **30 đoạn giữ riêng** | **30** | **chưa từng đo được** |

### Câu hỏi notebook này trả lời
1. Thêm dữ liệu câu thật có nâng được chrF++ trên dev-câu không?
2. **Model xử lý được văn bản dài đến đâu?** — dev-dài là lần đầu đo được.
3. Đường suy giảm theo độ dài trông thế nào?

### Mốc để so
- zero-shot **4,23** · fine-tune chỉ từ điển **34,85** (notebook 04)

---
### Cấu hình Kaggle
- **GPU T4** — không dùng P100 (sm_60, torch 2.10 đã bỏ)
- **Internet On** · dataset + notebook để **Private**
- **Save & Run All (Commit)** — `/kaggle/working` bị xoá khi hết session

### Khác notebook 03/04 ở đâu
`MAX_LEN` nâng **64 → 192**: đoạn dài nhất 83 âm tiết ≈ 95 token, để 64 sẽ
**cắt cụt** cả nguồn lẫn đích. Batch giảm 16 → 8 (accum 4, batch hiệu dụng vẫn 32)
để không tràn VRAM khi một batch rơi vào nhiều đoạn dài.


In [ ]:
# ==== CELL ĐẦU TIÊN, trước mọi import torch ====
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import torch
print('GPU:', torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.device_count() <= 1, 'vẫn >1 GPU -> Restart kernel'
if torch.cuda.is_available():
    arch = 'sm_%d%d' % torch.cuda.get_device_capability(0)
    archs = torch.cuda.get_arch_list()
    print('torch', torch.__version__, '| cần', arch, '| build:', archs)
    assert any(arch in a for a in archs), 'torch không có kernel cho %s -> đổi sang T4' % arch
    _ = (torch.ones(8, device='cuda') * 2).sum().item()
    print('kernel GPU: OK')


In [ ]:
!pip -q install --no-deps peft
!pip -q install sacrebleu
!pip -q uninstall -y torchao 2>/dev/null
import peft, transformers
print('peft', peft.__version__, '| transformers', transformers.__version__)


In [ ]:
QUIET = True
def show(x, n=40):
    x = str(x)
    return ('<%d ký tự / %d âm tiết>' % (len(x), len(x.split()))) if QUIET else x[:n]

import sys, io, json, glob, time, gc, re, unicodedata as ud
import numpy as np, pandas as pd
pd.set_option('display.max_columns', None); pd.set_option('display.width', 250)

CAND = []
for m in ('syllables.json','dict_pairs.csv','tv_norm.py'):
    CAND += glob.glob('/kaggle/input/**/' + m, recursive=True)
if not CAND:
    for t in sorted(glob.glob('/kaggle/input/*')):
        print(t, '->', sorted(os.listdir(t))[:14])
    raise SystemExit('Add Input -> taiviet-vi-mt (bản MỚI có train_sentences.csv), rồi Save Version.')
DATA = os.path.dirname(CAND[0]); WORK = '/kaggle/working'
MODEL_ID = 'facebook/nllb-200-distilled-600M'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
sys.path.insert(0, DATA)
import tv_norm
from tv_norm import syllables as tv_syllables, is_tv

with io.open(os.path.join(DATA,'syllables.json'), encoding='utf-8') as f:
    syl_meta = json.load(f)
SYL, CHARS = syl_meta['syllables'], syl_meta['char_fallback']
dict_pairs = pd.read_csv(os.path.join(DATA,'dict_pairs.csv'))
real_pairs = pd.read_csv(os.path.join(DATA,'real_pairs.csv'))

need = ['train_sentences.csv','train_paragraphs.csv','dev_paragraphs.csv']
miss = [f for f in need if not os.path.exists(os.path.join(DATA, f))]
if miss:
    raise SystemExit('Thiếu %s trong dataset.\n'
                     'Chạy scripts/ingest_translated.py rồi upload lại:\n'
                     '  kaggle datasets version -p kaggle_dataset -m "..." --dir-mode zip' % miss)
tr_sent = pd.read_csv(os.path.join(DATA,'train_sentences.csv'))
tr_para = pd.read_csv(os.path.join(DATA,'train_paragraphs.csv'))
dv_para = pd.read_csv(os.path.join(DATA,'dev_paragraphs.csv'))

print('DATA =', DATA, '|', DEV)
print('train: %d cụm từ điển + %d câu + %d đoạn = %d'
      % (len(dict_pairs), len(tr_sent), len(tr_para),
         len(dict_pairs)+len(tr_sent)+len(tr_para)))
print('dev  : %d câu | %d đoạn (dev-dài)' % (len(real_pairs), len(dv_para)))

ZS, NB04 = 4.23, 34.85          # mốc từ notebook 01 và 04


## Bước 1 — Dựng lại tokenizer, tự bổ sung âm tiết mới

Dữ liệu người bản ngữ dịch có thể chứa âm tiết chưa có trong kho (đo được
**107 loại, 1,8% token**). Notebook này **tự phát hiện và thêm** chúng, nên không
phụ thuộc việc kho đã cập nhật hay chưa.

Vì số token thêm vào giờ thay đổi theo dữ liệu, `assert` không còn cố định ở 4.678.
Hai điều kiện thật sự quan trọng vẫn được giữ nguyên: **`unk` = 0** và
**phục hồi âm tiết = 100%**.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tokenizers import AddedToken
from collections import Counter

# gom mọi âm tiết xuất hiện trong TOÀN BỘ dữ liệu
seen = Counter()
for df, col in ((dict_pairs,'tv'), (real_pairs,'tv'), (tr_sent,'tv'),
                (tr_para,'tv'), (dv_para,'tv')):
    for t in df[col].astype(str):
        seen.update(tv_syllables(t))
extra = sorted(s for s in seen if s not in SYL)
print('âm tiết trong kho: %d | xuất hiện trong dữ liệu: %d | NGOÀI kho: %d (%d token)'
      % (len(SYL), len(seen), len(extra), sum(seen[s] for s in extra)))

tok = AutoTokenizer.from_pretrained(MODEL_ID, src_lang='vie_Latn', tgt_lang='vie_Latn')
ORIG = len(tok)
def already(t):
    i = tok(t, add_special_tokens=False).input_ids
    return len(i) == 1 and i[0] != tok.unk_token_id

new_syl = [s for s in list(SYL) + extra if not already(s)]
new_chars = [c for c in CHARS if not already(c)]
NEW = new_syl + new_chars
n_added = tok.add_tokens([AddedToken(t, lstrip=True, rstrip=False,
                                     normalized=False, single_word=False) for t in NEW])
print('vocab %d -> %d (thêm %d)' % (ORIG, len(tok), n_added))

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
old_emb = model.get_input_embeddings().weight.data.clone()
model.resize_token_embeddings(len(tok))
emb = model.get_input_embeddings().weight.data
TN = old_emb[:ORIG].norm(dim=1).median().item()
UNK = tok.unk_token_id

def gv(gl):
    vs = []
    for x in gl:
        ids = [i for i in tok(str(x), add_special_tokens=False).input_ids
               if i < ORIG and i != UNK]
        if ids: vs.append(old_emb[ids].mean(0))
    return torch.stack(vs).mean(0) if vs else None

init_from, done = {}, {}
for s in new_syl:
    m = SYL.get(s)
    v = None
    if m:
        v = gv(m['direct']) if m['direct'] else (gv(m['indirect']) if m['indirect'] else None)
        if v is not None: init_from[s] = 'direct' if m['direct'] else 'indirect'
    if v is not None: done[s] = v
GLOBAL = torch.stack(list(done.values())).mean(0)
g = torch.Generator().manual_seed(0)
for s in NEW:
    if s not in done:
        done[s] = GLOBAL + 0.01 * TN * torch.randn(emb.shape[1], generator=g)
        init_from[s] = 'none'
for s, v in done.items():
    emb[tok.convert_tokens_to_ids(s)] = v / v.norm() * TN
print('nguồn khởi tạo:', dict(Counter(init_from.values())))

# --- kiểm tokenize trên TẤT CẢ dữ liệu, kể cả đoạn dài
SYLSET = set(list(SYL) + extra) | set(CHARS)
def check(texts, name):
    ids = [tok(str(t), add_special_tokens=False).input_ids for t in texts]
    nu = sum(sum(1 for j in i if j == UNK) for i in ids)
    rec = [[x for x in tok.convert_ids_to_tokens(i) if x in SYLSET] for i in ids]
    goc = [tv_syllables(str(t)) for t in texts]
    ok = sum(1 for a, b in zip(rec, goc) if a == b)
    nt = sum(len(i) for i in ids); ns = sum(len(x) for x in goc)
    print('  %-22s token %6d | unk %d | khớp %d/%d | token/âm tiết %.2f | max token %d'
          % (name, nt, nu, ok, len(texts), nt/max(1,ns), max(len(i) for i in ids)))
    return nu, ok, len(texts), max(len(i) for i in ids)

print('\nkiểm tokenize:')
res = [check(dict_pairs['tv'], 'cụm từ điển'),
       check(real_pairs['tv'], 'dev-câu (226)'),
       check(tr_sent['tv'],    'câu train (996)'),
       check(tr_para['tv'],    'đoạn train (115)'),
       check(dv_para['tv'],    'dev-dài (30)')]
assert all(r[0] == 0 for r in res), 'CÒN <unk> -> kho ký tự chưa phủ hết'
MAXTOK = max(r[3] for r in res)

# Vài dòng lẻ có thể không phục hồi được chuỗi âm tiết (lỗi gõ, ký tự lạ, dính
# chữ...). Một dòng hỏng KHÔNG nên chặn cả lần train — nhưng cũng không được
# âm thầm đưa vào. Cách xử lý: LOẠI chúng, báo rõ, và chỉ dừng nếu tỉ lệ đáng kể.
def bad_rows(df, col='tv'):
    out = []
    for i, t in enumerate(df[col].astype(str)):
        ids = tok(t, add_special_tokens=False).input_ids
        rec = [x for x in tok.convert_ids_to_tokens(ids) if x in SYLSET]
        if rec != tv_syllables(t):
            out.append(i)
    return out

DROP = {}
for nm, df in (('câu train', tr_sent), ('đoạn train', tr_para),
               ('dev-câu', real_pairs), ('dev-dài', dv_para)):
    b = bad_rows(df)
    if b:
        DROP[nm] = b
        print('  LOẠI %d/%d dòng ở %s (không phục hồi được chuỗi âm tiết)'
              % (len(b), len(df), nm))
        for i in b[:3]:
            idc = df.iloc[i].get('id', i)
            print('     %s: %s' % (idc, show(df['tv'].iloc[i])))
        assert len(b) / len(df) < 0.01,             '%s: %.1f%% dòng hỏng -> quá nhiều, phải xem lại dữ liệu' % (nm, 100*len(b)/len(df))

if DROP:
    tr_sent = tr_sent.drop(tr_sent.index[DROP.get('câu train', [])]).reset_index(drop=True)
    tr_para = tr_para.drop(tr_para.index[DROP.get('đoạn train', [])]).reset_index(drop=True)
    real_pairs = real_pairs.drop(real_pairs.index[DROP.get('dev-câu', [])]).reset_index(drop=True)
    dv_para = dv_para.drop(dv_para.index[DROP.get('dev-dài', [])]).reset_index(drop=True)
    print('  còn lại: %d câu | %d đoạn train | %d dev-câu | %d dev-dài'
          % (len(tr_sent), len(tr_para), len(real_pairs), len(dv_para)))
else:
    print('  không dòng nào phải loại.')

print('\nĐẠT. Token dài nhất trong toàn bộ dữ liệu: %d' % MAXTOK)


## Bước 2 — Dữ liệu, và vì sao `MAX_LEN` phải tăng

Đoạn dài nhất khoảng 83 âm tiết. Với `MAX_LEN=64` như notebook 03/04 thì
**cả nguồn lẫn đích đều bị cắt cụt** — model sẽ học từ dữ liệu què.


In [ ]:
from torch.utils.data import Dataset
from transformers import DataCollatorForSeq2Seq

MAX_LEN = 192
assert MAX_LEN > MAXTOK + 8, 'MAX_LEN %d quá ngắn cho token dài nhất %d' % (MAX_LEN, MAXTOK)

class Pairs(Dataset):
    def __init__(self, vi, tv):
        self.src = [str(x) for x in tv]      # nguồn = chữ Thái
        self.tgt = [str(x) for x in vi]      # đích  = tiếng Việt
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        m = tok(self.src[i], text_target=self.tgt[i], max_length=MAX_LEN, truncation=True)
        return {'input_ids': m['input_ids'], 'attention_mask': m['attention_mask'],
                'labels': m['labels']}

train_ds = Pairs(
    list(dict_pairs['vi']) + list(tr_sent['vi']) + list(tr_para['vi']),
    list(dict_pairs['tv']) + list(tr_sent['tv']) + list(tr_para['tv']))
dev_sent = Pairs(real_pairs['vi'], real_pairs['tv'])
dev_long = Pairs(dv_para['vi'], dv_para['tv'])
print('train %d | dev-câu %d | dev-dài %d' % (len(train_ds), len(dev_sent), len(dev_long)))

L = [len(train_ds[i]['input_ids']) for i in range(0, len(train_ds), 20)]
print('độ dài nguồn train (mẫu): trung vị %d, p95 %d, max %d'
      % (int(np.median(L)), int(np.percentile(L, 95)), max(L)))
collator = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100)


In [ ]:
from peft import LoraConfig, get_peft_model
import torch.nn.functional as F, sacrebleu, inspect
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, TrainerCallback)

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='SEQ_2_SEQ_LM',
    target_modules=['q_proj','k_proj','v_proj','out_proj','fc1','fc2']))

# BẪY 1 — LoRA không train embedding; mask gradient để CHỈ token mới cập nhật
w = model.get_input_embeddings().weight
w.requires_grad_(True)
w.register_hook(lambda gr: torch.cat([torch.zeros_like(gr[:ORIG]), gr[ORIG:]], 0))
assert w.requires_grad, 'embedding vẫn đóng băng!'

# BẪY 2 — lm_head tie: chặn token chữ Thái ở đầu ra
V = model.get_input_embeddings().weight.shape[0]
BAD = torch.zeros(V, dtype=torch.bool, device=DEV)
BAD[torch.tensor([tok.convert_tokens_to_ids(t) for t in NEW])] = True
print('vocab: tokenizer %d | model %d | token bị chặn %d' % (len(tok), V, int(BAD.sum())))
model = model.to(DEV)
emb_before = model.get_input_embeddings().weight.data[ORIG:].clone().cpu()

chrf = sacrebleu.CHRF(word_order=2)

# --- THÊM CALLBACK NÀY ĐỂ BẮT DỮ LIỆU LOSS CHUẨN ---
class LossCallback(TrainerCallback):
    def __init__(self):
        self.loss_trace = []
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.loss_trace.append(logs["loss"])

loss_callback = LossCallback()

def metrics(p):
    pr, lb = p
    pr = np.where(pr != -100, pr, tok.pad_token_id)
    lb = np.where(lb != -100, lb, tok.pad_token_id)
    h = tok.batch_decode(pr, skip_special_tokens=True)
    r = tok.batch_decode(lb, skip_special_tokens=True)
    return {'chrf2': round(chrf.corpus_score(h, [r]).score, 2),
            'output_khac_nhau': len(set(h))}

_K = 'eval_strategy' if 'eval_strategy' in inspect.signature(
    Seq2SeqTrainingArguments.__init__).parameters else 'evaluation_strategy'

args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(WORK,'nb05'), **{_K:'epoch'},
    per_device_train_batch_size=4,        
    gradient_accumulation_steps=8,        
    per_device_eval_batch_size=4,         
    gradient_checkpointing=False,          
    learning_rate=2e-4, num_train_epochs=10,
    warmup_ratio=0.05, weight_decay=0.0, max_grad_norm=1.0,
    lr_scheduler_type='cosine', fp16=(DEV=='cuda'),
    save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='chrf2', greater_is_better=True,
    save_total_limit=1, logging_steps=50, report_to=[], seed=0,
    predict_with_generate=True, generation_max_length=MAX_LEN,
    generation_num_beams=1,               
    label_names=['labels'])

model.generation_config.forced_bos_token_id = tok.convert_tokens_to_ids('vie_Latn')
# CHẶN TOKEN BẰNG CONFIG CHUẨN CỦA NLLB (Không can thiệp loss thủ công)
model.generation_config.suppress_tokens = BAD.nonzero().flatten().tolist()

trainer = Seq2SeqTrainer(
    model=model, 
    args=args, 
    train_dataset=train_ds,
    eval_dataset=dev_sent,
    data_collator=collator, 
    compute_metrics=metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3), loss_callback]
)
print('sẵn sàng. Chọn model theo dev-CÂU (như notebook 03/04) để so sánh công bằng.')

In [ ]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

t0 = time.time(); trainer.train(); mins = (time.time()-t0)/60
print('train xong sau %.1f phút' % mins)

after = model.get_input_embeddings().weight.data[ORIG:].clone().cpu()
d_new = float((after - emb_before).norm(dim=1).median())
d_old = float((model.get_input_embeddings().weight.data[:ORIG].cpu() - old_emb[:ORIG]).norm(dim=1).max())
print('dịch chuyển embedding: mới %.4f | gốc %.4f' % (d_new, d_old))
assert d_new > 1e-4, 'EMBEDDING MỚI KHÔNG HỌC -> bẫy 1!'
assert d_old < 1e-4, 'embedding GỐC bị đổi -> mask gradient sai!'
lt = loss_callback.loss_trace
print('loss thật: n=%d, đầu %.3f, cuối %.3f' % (len(lt), lt[0], lt[-1]))

## Bước 3 — Hai phép đo, và đường suy giảm theo độ dài

Đây là phần trả lời câu hỏi chính. `dev-dài` chưa từng đo được ở notebook nào trước.


In [ ]:
ev_c = trainer.evaluate(dev_sent, metric_key_prefix='cau')
ev_d = trainer.evaluate(dev_long, metric_key_prefix='dai')   # đo RIÊNG
ev = {**ev_c, **ev_d}
sent_chrf, long_chrf = ev['cau_chrf2'], ev['dai_chrf2']
print(json.dumps({k:v for k,v in ev.items() if isinstance(v,(int,float))},
                 ensure_ascii=False, indent=1))
print()
print('DEV-CÂU (226) : chrF++ %.2f   | notebook 04 đạt %.2f  -> %+.2f'
      % (sent_chrf, NB04, sent_chrf - NB04))
print('DEV-DÀI (30)  : chrF++ %.2f' % long_chrf)
print('chênh lệch câu vs đoạn: %.2f điểm' % (sent_chrf - long_chrf))
print('loss: câu %.3f | đoạn %.3f  <- loss cao hơn = đoạn KHÓ hơn thật,'
      % (ev['cau_loss'], ev['dai_loss']))
print('      dù chrF++ gần bằng nhau. chrF++ dễ dãi hơn với văn bản dài.')
print('output khác nhau: câu %d/%d | đoạn %d/%d'
      % (ev['cau_output_khac_nhau'], len(dev_sent),
         ev['dai_output_khac_nhau'], len(dev_long)))


In [ ]:
# ĐƯỜNG SUY GIẢM THEO ĐỘ DÀI — gộp cả hai tập dev rồi chia nhóm
def gen(ds, bs=8):
    out = []
    for i in range(0, len(ds), bs):
        b = [ds[j] for j in range(i, min(i+bs, len(ds)))]
        enc = collator(b)
        # collator tạo cả decoder_input_ids từ labels; truyền vào generate() là
        # SAI vì generate tự dựng lấy. Bỏ cả hai.
        enc = {k: v.to(DEV) for k, v in enc.items()
               if k not in ('labels', 'decoder_input_ids')}
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=MAX_LEN, num_beams=4)
        out += tok.batch_decode(g, skip_special_tokens=True)
    return out

pool_src = list(real_pairs['tv']) + list(dv_para['tv'])
pool_ref = list(real_pairs['vi']) + list(dv_para['vi'])
hyp = gen(dev_sent) + gen(dev_long)
nsyl = [len(tv_syllables(str(t))) for t in pool_src]

def bucket(n):
    return '1-5' if n <= 5 else '6-10' if n <= 10 else '11-20' if n <= 20 \
        else '21-35' if n <= 35 else '36+'
rows = []
for b in ['1-5','6-10','11-20','21-35','36+']:
    idx = [i for i, n in enumerate(nsyl) if bucket(n) == b]
    if not idx: continue
    h = [hyp[i] for i in idx]; r = [pool_ref[i] for i in idx]
    rows.append(dict(am_tiet=b, n=len(idx),
                     chrf2=round(chrf.corpus_score(h, [r]).score, 2),
                     output_khac_nhau=len(set(h)),
                     do_dai_tb=round(float(np.mean([nsyl[i] for i in idx])), 1)))
deg = pd.DataFrame(rows)
display(deg)
print('\nĐọc bảng: chrF++ giảm bao nhiêu khi câu dài ra? Đó là hình cho bài báo.')
print('Nếu nhóm 36+ tụt mạnh -> vẫn cần thêm dữ liệu đoạn văn.')


In [ ]:
outdir = os.path.join(WORK,'nb05_final')
model.save_pretrained(outdir); tok.save_pretrained(os.path.join(WORK,'tok_ext_v2'))
torch.save({'new_embeddings': model.get_input_embeddings().weight.data[ORIG:].cpu(),
            'orig_vocab': ORIG, 'new_tokens': NEW},
           os.path.join(WORK,'new_embeddings_nb05.pt'))

rep = dict(setup='train = từ điển + 996 câu + 115 đoạn',
           train_n=len(train_ds), n_added=n_added, max_len=MAX_LEN,
           dev_cau=dict(n=len(dev_sent), chrf2=sent_chrf,
                        output_khac_nhau=int(ev['cau_output_khac_nhau'])),
           dev_dai=dict(n=len(dev_long), chrf2=long_chrf,
                        output_khac_nhau=int(ev['dai_output_khac_nhau'])),
           moc=dict(zero_shot=ZS, nb04_chi_tu_dien=NB04),
           emb_delta_new=d_new, emb_delta_old=d_old,
           loss_dau=round(lt[0],3), loss_cuoi=round(lt[-1],3), phut=round(mins,1),
           suy_giam=deg.to_dict('records'))
with open(os.path.join(WORK,'report_05.json'),'w',encoding='utf-8') as f:
    json.dump(rep, f, ensure_ascii=False, indent=1, default=str)
deg.to_csv(os.path.join(WORK,'do_dai_chrf.csv'), index=False)
print(json.dumps(rep, ensure_ascii=False, indent=1, default=str))
print('\nNHỚ: Clear all outputs -> Save & Run All (Commit).')
print('\nCòn thiếu để hoàn thành bài báo:')
print('  - TẬP TEST từ văn bản sách báo thật, chia theo tài liệu gốc')
print('  - 3.000 cặp câu song ngữ đang chuẩn bị')
print('  - trần con người: cho 2 người dịch trùng ~200 câu')


In [ ]:
import glob, os

checkpoints = glob.glob('/kaggle/working/nb05/checkpoint-*')

if checkpoints:
    latest_cp = sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1]
    print(f"--> Đang load checkpoint để train tiếp: {latest_cp}")
    # SỬA LẠI CHỖ NÀY: Thêm dấu =
    trainer.train(resume_from_checkpoint=latest_cp)
else:
    print("--> Không tìm thấy checkpoint, train từ đầu")
    trainer.train()

In [ ]:
import pandas as pd

logs = trainer.state.log_history
epoch_data = {}

for log in logs:
    ep = round(log.get('epoch', 0))
    if ep not in epoch_data:
        epoch_data[ep] = {}
    
    if 'loss' in log:
        epoch_data[ep]['Training Loss'] = log['loss']
    if 'eval_loss' in log:
        epoch_data[ep]['Validation Loss'] = log['eval_loss']
    if 'eval_chrf2' in log:
        epoch_data[ep]['Chrf2'] = log['eval_chrf2']
    if 'eval_output_khac_nhau' in log:
        epoch_data[ep]['Output Khác Nhau'] = log['eval_output_khac_nhau']

# Tạo DataFrame chính
df = pd.DataFrame.from_dict(epoch_data, orient='index').reset_index()
df = df.rename(columns={'index': 'Epoch'}).sort_values('Epoch')
df_clean = df.dropna(subset=['Chrf2']).copy()

# 1. DÒNG TỔNG HỢP BEST (Giá trị tốt nhất)
best_row = {
    'Epoch': 'TỔNG KẾT (BEST)',
    'Training Loss': df_clean['Training Loss'].min() if 'Training Loss' in df_clean else None, # Loss train thấp nhất
    'Validation Loss': df_clean['Validation Loss'].min(), # Loss val thấp nhất
    'Chrf2': df_clean['Chrf2'].max(),                      # Chrf2 cao nhất
    'Output Khác Nhau': df_clean['Output Khác Nhau'].max() if 'Output Khác Nhau' in df_clean else None
}

# 2. DÒNG TỔNG HỢP TRUNG BÌNH (Mean / Average)
avg_row = {
    'Epoch': 'TỔNG KẾT (TRUNG BÌNH)',
    'Training Loss': df_clean['Training Loss'].mean() if 'Training Loss' in df_clean else None,
    'Validation Loss': df_clean['Validation Loss'].mean(),
    'Chrf2': df_clean['Chrf2'].mean(),
    'Output Khác Nhau': df_clean['Output Khác Nhau'].mean() if 'Output Khác Nhau' in df_clean else None
}

# Nối 2 dòng tổng kết vào cuối bảng
df_final = pd.concat([df_clean, pd.DataFrame([best_row, avg_row])], ignore_index=True)

cols = ['Epoch', 'Training Loss', 'Validation Loss', 'Chrf2', 'Output Khác Nhau']
cols = [c for c in cols if c in df_final.columns]

# Làm tròn các con số thập phân cho dễ nhìn
numeric_cols = ['Training Loss', 'Validation Loss', 'Chrf2']
df_final[numeric_cols] = df_final[numeric_cols].round(4)

print("=== LỊCH SỬ HUẤN LUYỆN + DÒNG BEST & TRUNG BÌNH ===")
display(df_final[cols])